# Realistic dataset

In [1]:
data_path = '../data/baselines/PatchFinder'
top100_csv_filename = "top100_test.csv" # Name for the output CSV file
pred_top100_csv_filename = "predictions_top100_test.csv" # Name for the output CSV file

In [2]:
import os
import polars as pl


top100_df = pl.read_csv(os.path.join(data_path, top100_csv_filename))
pred_top100_df = pl.read_csv(os.path.join(data_path, pred_top100_csv_filename))


top100_df = (
    top100_df
    .join(
        pred_top100_df,
        on=set(top100_df.columns).intersection(set(pred_top100_df.columns)),
        how="left"
    )
)
top100_df

cve,repo,commit_id,similarity,label,desc_token,msg_token,diff_token,recall,precision,f1,fused_f1,prediction
str,str,str,f64,i64,str,str,str,f64,f64,f64,f64,f64
"""CVE-2005-2351""","""muttmua/mutt""","""4927240d1f8fca0806df1e48e5b09e…",0.047721,0,"""Mutt before 1.5.20 patch 7 all…","""automatic post-release commit …","""diff -- git a/ChangeLog b/Chan…",0.301331,0.50792,0.378256,0.425977,0.000445
"""CVE-2005-2351""","""muttmua/mutt""","""a327386c5bf676a8321335fca84915…",0.053276,0,"""Mutt before 1.5.20 patch 7 all…","""Stable branch quick fix for pa…","""diff -- git a/curs_main.c b/cu…",0.311758,0.463529,0.372788,0.426064,0.000006
"""CVE-2005-2351""","""muttmua/mutt""","""f30dd804942eee1339d59785cde4d0…",0.055703,0,"""Mutt before 1.5.20 patch 7 all…","""This small patch : * catches a…","""diff -- git a/curs_main.c b/cu…",0.286883,0.52324,0.370582,0.426285,0.000005
"""CVE-2005-2351""","""muttmua/mutt""","""e313244657623f23e5d9a8da698a98…",0.042647,0,"""Mutt before 1.5.20 patch 7 all…","""# changelog commit""","""diff -- git a/ChangeLog b/Chan…",0.327086,0.464208,0.383766,0.426413,0.000004
"""CVE-2005-2351""","""muttmua/mutt""","""848834424235c850947c164ff782d4…",0.065582,0,"""Mutt before 1.5.20 patch 7 all…","""Fix next_token ( ) oob read . …","""diff -- git a/rfc822.c b/rfc82…",0.287244,0.485146,0.360842,0.426424,0.00068
…,…,…,…,…,…,…,…,…,…,…,…,…
"""CVE-2022-35411""","""abersheeran/rpc.py""","""24a2781b73f8d500d8405682e45810…",0.175148,0,"""rpc.py through 0.6.0 allows Re…","""Update description""","""diff -- git a/README.md b/READ…",0.426604,0.5624,0.485179,0.660327,0.00001
"""CVE-2022-35411""","""abersheeran/rpc.py""","""a04fa00d8bcee13ec540b3e42758a6…",0.222159,0,"""rpc.py through 0.6.0 allows Re…","""More tests""","""diff -- git a/rpcpy/utils/open…",0.403545,0.508194,0.449864,0.672023,0.000016
"""CVE-2022-35411""","""abersheeran/rpc.py""","""3859504738df9c9088ad4ba81fc542…",0.217127,0,"""rpc.py through 0.6.0 allows Re…","""Update README.md""","""diff -- git a/README.md b/READ…",0.440322,0.548692,0.48857,0.705697,0.000009


In [4]:
import re

def find_cve_ids(text):
    """
    Finds all CVE ID references in a given text.
    
    Args:
        text (str): The input string to search for CVE IDs.
    
    Returns:
        list: A list of found CVE IDs.
    """
    cve_pattern = r'CVE-\d{4}-\d{4,}'
    return re.findall(cve_pattern, text)


In [ ]:
from datasets import load_dataset

patches_ds = load_dataset("andstor/cvevc_commits", "patches", split="test")

#explicit_patches = ddict.filter(lambda x: find_cve_ids(x['commit_message']), num_proc=10)
#implicit_patches = patches_ds.filter(lambda x: not find_cve_ids(x['commit_message']), num_proc=10)

In [42]:
from datasets import get_dataset_infos

infos = get_dataset_infos("andstor/cvevc_commits")

top_100_commit_ids = top100_df['commit_id'].to_list()
top_100_commit_data = []

In [ ]:
from tqdm import tqdm

non_patches_ds = load_dataset("andstor/cvevc_commits", "non_patches", split="test", streaming=True)
num_examples = infos["non_patches"].splits["test"].num_examples

for row in tqdm(non_patches_ds, total=num_examples):
    # Check if the row is in the top 100 DataFrame
    if row['commit_id'] in top_100_commit_ids:
        # Append the row to the top_100_commit_data list
        top_100_commit_data.append(row)
        


100%|██████████| 2150904/2150904 [1:05:39<00:00, 545.96it/s]


In [43]:
from tqdm import tqdm
from datasets import get_dataset_infos

patches_ds = load_dataset("andstor/cvevc_commits", "patches", split="test")
num_examples = infos["patches"].splits["test"].num_examples

for row in tqdm(patches_ds, total=num_examples):
    # Check if the row is in the top 100 DataFrame
    if row['commit_id'] in top_100_commit_ids:
        # Append the row to the top_100_commit_data list
        top_100_commit_data.append(row)
        


100%|██████████| 1453/1453 [00:01<00:00, 842.68it/s]


In [8]:
import pickle
# Save the top_100_commit_data list to a pickle file
with open('top_100_commit_data.pkl', 'wb') as f:
    pickle.dump(top_100_commit_data, f)

In [15]:
import pickle
with open('top_100_commit_data.pkl', 'rb') as f:
    top_100_commit_data = pickle.load(f)

In [16]:
import polars as pl
top_100_commit_df = pl.DataFrame(top_100_commit_data)
top_100_commit_df


commit_id,repo,commit_message,diff,label
str,str,str,str,i64
"""388d0e6d391482ea8e8691efef8ef8…","""02strich/pykerberos""","""Added missing error messge ""","""commit 388d0e6d391482ea8e8691e…",0
"""481fc62b286fc2f1a5e8286e402713…","""02strich/pykerberos""","""Merge pull request #23 from di…","""commit 481fc62b286fc2f1a5e8286…",0
"""5867201f1b9c682402aa9b495a654b…","""02strich/pykerberos""","""Adding missing optional marker…","""commit 5867201f1b9c682402aa9b4…",0
"""39ec626539c355940cb2908d23e35a…","""02strich/pykerberos""","""add winrm-style IOV encryption…","""commit 39ec626539c355940cb2908…",0
"""17aeaef915e197a371eb49a4f9b6b8…","""02strich/pykerberos""","""v1.2.4 release Changelog and …","""commit 17aeaef915e197a371eb49a…",0
…,…,…,…,…
"""395830d43349ed5bc0633c3f44c6cd…","""zeit/serve""","""Rewrite project entirely (#374…","""commit 395830d43349ed5bc0633c3…",1
"""c273f40a413d7f65abb8bf0f30bf14…","""zopefoundation/AccessControl""","""LP #1047318: Tighten import re…","""commit c273f40a413d7f65abb8bf0…",1
"""b42dd4badf803bb9fb71ac34cd9cb0…","""zopefoundation/AccessControl""","""Merge pull request from GHSA-q…","""commit b42dd4badf803bb9fb71ac3…",1


In [17]:

cands_df = (
    top100_df
    .join(
        top_100_commit_df,
        on=['repo', 'commit_id', 'label',],
        how="inner"
    )
    .with_columns([
        (-pl.col("prediction")).rank("ordinal").over("cve").alias("rank")  # rank 1 = best
    ])
    .select(['cve', 'repo', 'commit_id', 'commit_message', 'diff', 'label', 'rank'])

)

cands_df

cve,repo,commit_id,commit_message,diff,label,rank
str,str,str,str,str,i64,u32
"""CVE-2005-2351""","""muttmua/mutt""","""4927240d1f8fca0806df1e48e5b09e…","""automatic post-release commit …","""commit 4927240d1f8fca0806df1e4…",0,7
"""CVE-2005-2351""","""muttmua/mutt""","""a327386c5bf676a8321335fca84915…","""Stable branch quick fix for pa…","""commit a327386c5bf676a8321335f…",0,51
"""CVE-2005-2351""","""muttmua/mutt""","""f30dd804942eee1339d59785cde4d0…","""This small patch: * catches a…","""commit f30dd804942eee1339d5978…",0,59
"""CVE-2005-2351""","""muttmua/mutt""","""e313244657623f23e5d9a8da698a98…","""# changelog commit ""","""commit e313244657623f23e5d9a8d…",0,92
"""CVE-2005-2351""","""muttmua/mutt""","""848834424235c850947c164ff782d4…","""Fix next_token() oob read. (c…","""commit 848834424235c850947c164…",0,6
…,…,…,…,…,…,…
"""CVE-2022-35411""","""abersheeran/rpc.py""","""24a2781b73f8d500d8405682e45810…","""Update description ""","""commit 24a2781b73f8d500d840568…",0,91
"""CVE-2022-35411""","""abersheeran/rpc.py""","""a04fa00d8bcee13ec540b3e42758a6…","""More tests ""","""commit a04fa00d8bcee13ec540b3e…",0,33
"""CVE-2022-35411""","""abersheeran/rpc.py""","""3859504738df9c9088ad4ba81fc542…","""Update README.md ""","""commit 3859504738df9c9088ad4ba…",0,94


In [18]:
from datasets import load_dataset
cve_ds = load_dataset("andstor/cvevc_cve", split="test")
cve_df = cve_ds.to_polars()


/Users/andrestorhaug/Code/Projects/agentic-security-patch-classification-replication-package/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<function random_array at 0x3d84127a0>
<function random at 0x3d8412ac0>
<function rand at 0x3d8412c00>


In [24]:
# Add cve data to the top 100 commit data"
cve_cands_df = (
    cands_df
    .join(
        cve_df,
        on=["cve"],
        how="inner"
    )
    .select(['cve', 'desc', 'repo', 'commit_id', 'commit_message', 'diff', 'label', 'rank'])
    .sort("cve")
)
cve_cands_df

cve,desc,repo,commit_id,commit_message,diff,label,rank
str,str,str,str,str,str,i64,u32
"""CVE-2005-2351""","""Mutt before 1.5.20 patch 7 all…","""muttmua/mutt""","""4927240d1f8fca0806df1e48e5b09e…","""automatic post-release commit …","""commit 4927240d1f8fca0806df1e4…",0,7
"""CVE-2005-2351""","""Mutt before 1.5.20 patch 7 all…","""muttmua/mutt""","""a327386c5bf676a8321335fca84915…","""Stable branch quick fix for pa…","""commit a327386c5bf676a8321335f…",0,51
"""CVE-2005-2351""","""Mutt before 1.5.20 patch 7 all…","""muttmua/mutt""","""f30dd804942eee1339d59785cde4d0…","""This small patch: * catches a…","""commit f30dd804942eee1339d5978…",0,59
"""CVE-2005-2351""","""Mutt before 1.5.20 patch 7 all…","""muttmua/mutt""","""e313244657623f23e5d9a8da698a98…","""# changelog commit ""","""commit e313244657623f23e5d9a8d…",0,92
"""CVE-2005-2351""","""Mutt before 1.5.20 patch 7 all…","""muttmua/mutt""","""848834424235c850947c164ff782d4…","""Fix next_token() oob read. (c…","""commit 848834424235c850947c164…",0,6
…,…,…,…,…,…,…,…
"""CVE-2022-35411""","""rpc.py through 0.6.0 allows Re…","""abersheeran/rpc.py""","""24a2781b73f8d500d8405682e45810…","""Update description ""","""commit 24a2781b73f8d500d840568…",0,91
"""CVE-2022-35411""","""rpc.py through 0.6.0 allows Re…","""abersheeran/rpc.py""","""a04fa00d8bcee13ec540b3e42758a6…","""More tests ""","""commit a04fa00d8bcee13ec540b3e…",0,33
"""CVE-2022-35411""","""rpc.py through 0.6.0 allows Re…","""abersheeran/rpc.py""","""3859504738df9c9088ad4ba81fc542…","""Update README.md ""","""commit 3859504738df9c9088ad4ba…",0,94


In [27]:
cve_cands_df["diff"].is_null().sum()

0

In [26]:
from datasets import Dataset
cve_cands_ds = Dataset.from_polars(cve_cands_df, split="test")
cve_cands_ds

Dataset({
    features: ['cve', 'desc', 'repo', 'commit_id', 'commit_message', 'diff', 'label', 'rank'],
    num_rows: 179473
})

In [28]:
cve_cands_ds.push_to_hub("andstor/cvevc_candidates", config_name="PatchFinder_top100", private=False, max_shard_size="250MB")

Creating parquet from Arrow format: 100%|██████████| 18/18 [00:00<00:00, 67.62ba/s]
Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

Processing Files (0 / 1)                :  30%|███       | 14.1MB / 46.8MB, 23.6MB/s  

Processing Files (0 / 1)                :  31%|███▏      | 14.7MB / 46.8MB, 14.7MB/s  
Processing Files (0 / 1)                :  33%|███▎      | 15.2MB / 46.8MB, 12.7MB/s  
Processing Files (0 / 1)                :  42%|████▏     | 19.4MB / 46.8MB, 13.9MB/s  
Processing Files (0 / 1)                :  47%|████▋     | 22.1MB / 46.8MB, 13.8MB/s  
Processing Files (0 / 1)                :  53%|█████▎    | 24.7MB / 46.8MB, 13.7MB/s  
Processing Files (0 / 1)                :  60%|█████▉    | 27.9MB / 46.8MB, 13.9MB/s  
Processing Files (0 / 1)                :  66%|██████▋   | 31.1MB / 46.8MB, 14.1MB/s  
Processing Files (0 / 1)                :  71%|███████   | 33.2MB / 46.8MB, 13.8MB/s  
Processing Files (0 / 1)                :  80%|█

CommitInfo(commit_url='https://huggingface.co/datasets/andstor/cvevc_candidates/commit/23d503befca0daab9dc03f7eeb738e06dd6cbcd5', commit_message='Upload dataset', commit_description='', oid='23d503befca0daab9dc03f7eeb738e06dd6cbcd5', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/andstor/cvevc_candidates', endpoint='https://huggingface.co', repo_type='dataset', repo_id='andstor/cvevc_candidates'), pr_revision=None, pr_num=None)

In [29]:
cve_cands_ds

Dataset({
    features: ['cve', 'desc', 'repo', 'commit_id', 'commit_message', 'diff', 'label', 'rank'],
    num_rows: 179473
})

In [30]:
cve_cands_ds_top10 = cve_cands_ds.filter(lambda x: x['rank'] <= 10)

Filter: 100%|██████████| 179473/179473 [00:01<00:00, 98695.43 examples/s] 


In [ ]:
cve_cands_ds_top10.push_to_hub("andstor/cvevc_candidates", config_name="PatchFinder_top10", private=False, max_shard_size="250MB")

Creating parquet from Arrow format: 100%|██████████| 13/13 [00:00<00:00, 16.81ba/s]
Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

Processing Files (0 / 1)                :  31%|███       | 15.2MB / 49.9MB,   ???B/s  
Processing Files (0 / 1)                :  35%|███▍      | 17.4MB / 49.9MB, 10.7MB/s  

Processing Files (0 / 1)                :  44%|████▍     | 22.2MB / 49.9MB, 11.6MB/s  
Processing Files (0 / 1)                :  50%|████▉     | 24.9MB / 49.9MB, 12.1MB/s  
Processing Files (0 / 1)                :  56%|█████▋    | 28.1MB / 49.9MB, 12.9MB/s  
Processing Files (0 / 1)                :  62%|██████▏   | 30.8MB / 49.9MB, 13.0MB/s  
Processing Files (0 / 1)                :  67%|██████▋   | 33.4MB / 49.9MB, 13.0MB/s  
Processing Files (0 / 1)                :  73%|███████▎  | 36.7MB / 49.9MB, 13.4MB/s  
Processing Files (0 / 1)                :  79%|███████▉  | 39.3MB / 49.9MB, 13.4MB/s  
Processing Files (0 / 1)                :  85%|█

CommitInfo(commit_url='https://huggingface.co/datasets/andstor/cvevc_candidates/commit/153fbc99f592ad5594c1b4607b4b556eb3caedf1', commit_message='Upload dataset', commit_description='', oid='153fbc99f592ad5594c1b4607b4b556eb3caedf1', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/andstor/cvevc_candidates', endpoint='https://huggingface.co', repo_type='dataset', repo_id='andstor/cvevc_candidates'), pr_revision=None, pr_num=None)

# Random dataset

In [ ]:
from datasets import load_dataset, DatasetDict, Dataset

ds_cve = load_dataset("andstor/cvevc_cve")
ds_patches = load_dataset("andstor/cvevc_commits", "patches")
ds_nonpatches = load_dataset("andstor/cvevc_commits", "non_patches")
ds_mappings = load_dataset("andstor/cvevc_cve_commit_mappings")


Resolving data files:   0%|          | 0/140 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/83 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/96 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/140 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/83 [00:00<?, ?it/s]

...

Resolving data files:   0%|          | 0/83 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/96 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/79 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/52 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/58 [00:00<?, ?it/s]

In [ ]:
from tqdm import tqdm
import pandas as pd

ddict_100 = DatasetDict()
ddict_10 = DatasetDict()

for split in ["test"]:
    # Index for commit_id lookup
    cindex = {key: idx for idx, key in tqdm(enumerate(ds_cve[split]["cve"]), total=len(ds_cve[split]["cve"]), desc=f"Indexing CVEs {split}")}
    pindex = {key: idx for idx, key in tqdm(enumerate(ds_patches[split]["commit_id"]), total=len(ds_patches[split]["commit_id"]), desc=f"Indexing patch commits {split}")}
    npindex = {key: idx for idx, key in tqdm(enumerate(ds_nonpatches[split]["commit_id"]), total=len(ds_nonpatches[split]["commit_id"]), desc=f"Indexing non-patch commits {split}")}
    
    
    def mapping_to_record(example):
        cve = example["cve"]
        commit_id = example["commit_id"]
        
        cve_row = ds_cve[split][cindex[cve]]
        commit_row = None
        if commit_id in pindex: # Patch commit
            commit_row = ds_patches[split][pindex[commit_id]]
        else: # Non-patch commit
            commit_row = ds_nonpatches[split][npindex[commit_id]]
        
        if commit_row is not None:
            return {
                "cve": cve,
                "desc": cve_row["desc"],
                "repo": commit_row["repo"],
                "commit_id": commit_id,
                "commit_message": commit_row["commit_message"],
                "diff": commit_row["diff"],
                "label": example["label"],
            }
        else:
            return None
    
    cve_mappings = ds_mappings[split].to_pandas().groupby("cve")
    
    subset_100 = []
    subset_10 = []
    for cve, group_df in tqdm(cve_mappings):
        # select all patches for this group_df. use label == 1
        cve_data = []
        patches = group_df[group_df["label"] == 1]
        # now select 10 - len(patches) non-patches


        non_patches_pool = group_df[group_df["label"] == 0]
        non_patches_pool = non_patches_pool.sample(min(len(non_patches_pool), 100), replace=False, random_state=42)

        needed = max(0, 100 - len(patches))
        non_patches = non_patches_pool[:needed]
        cve_df = pd.concat([patches, non_patches])
        cve_df = cve_df.apply(mapping_to_record, axis=1, result_type='expand')
        subset_100.append(cve_df)
        
        needed = max(0, 10 - len(patches))
        non_patches = non_patches_pool[:needed]
        cve_df = pd.concat([patches, non_patches])
        cve_df = cve_df.apply(mapping_to_record, axis=1, result_type='expand')
        subset_10.append(cve_df)

    final_100 = pd.concat(subset_100, ignore_index=True)
    final_10 = pd.concat(subset_10, ignore_index=True)
    ddict_100[split] = Dataset.from_pandas(final_100)
    ddict_10[split] = Dataset.from_pandas(final_10)


 74%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▊                                                    | 5970/8019 [03:58<02:29, 13.73it/s]

In [ ]:
ddict_10.push_to_hub("andstor/cvevc_candidates", "random_10", private=False, max_shard_size="250MB")
ddict_100.push_to_hub("andstor/cvevc_candidates", "random_100", private=False, max_shard_size="250MB")